# Step by Step Guide to run models on Raspberry Pi Zero 2W

- Import models
- Create a Virtual environment
- Activate it
- pip install jupyter
- run jupyter notebook --ip=[YOUR IP ADDREES] --no-browser
- Open on Browser or VSCode with link

## Teste Microfone & Python:

In [ ]:
!pwd

In [ ]:
!arecord -l

In [ ]:
!ls

In [ ]:
!sudo apt-get install portaudio19-dev python3-dev -y

In [ ]:
!pip install pyaudio

In [ ]:
import pyaudio

p = pyaudio.PyAudio()
info = p.get_host_api_info_by_index(0)
numdevices = info.get('deviceCount')

print("Dispositivos de áudio encontrados:")
for i in range(0, numdevices):
    device_info = p.get_device_info_by_host_api_device_index(0, i)
    if device_info.get('maxInputChannels') > 0:
        print(f"  Índice de Entrada {i}: {device_info.get('name')}")

p.terminate()

In [ ]:
!pip install numpy==1.25.2

In [ ]:
!pip install scipy

In [ ]:
import pyaudio
import wave
import numpy as np
print(np.__version__)
from scipy.signal import resample_poly

FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 48000
CHUNK = 1024
RECORD_SECONDS = 10
DEVICE_INDEX = 0   # replace this with your detected USB mic's index
WAVE_OUTPUT_FILENAME = "output.wav"

TARGET_RATE = 16000

audio = pyaudio.PyAudio()

stream = audio.open(format=FORMAT, channels=CHANNELS,
                    rate=RATE, input=True, input_device_index=DEVICE_INDEX,
                    frames_per_buffer=CHUNK)

print("Recording...")
frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK, exception_on_overflow=False)
    frames.append(data)

print("Finished recording.")

stream.stop_stream()
stream.close()
audio.terminate()

audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# Calcula os fatores para a reamostragem (de 48k para 16k é 1/3)
resampled_data = resample_poly(audio_data, TARGET_RATE, RATE)

# Converte de volta para int16
resampled_data = resampled_data.astype(np.int16)

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')

# Define os parâmetros do ÁUDIO REAMOSTRADO
wf.setnchannels(CHANNELS)
wf.setsampwidth(audio.get_sample_size(FORMAT))
wf.setframerate(TARGET_RATE) # <-- MUDANÇA 1: Usar a taxa de amostragem de destino

# Escreve os dados REAMOSTRADOS no arquivo
# .tobytes() converte o array NumPy de volta para bytes
wf.writeframes(resampled_data.tobytes()) # <-- MUDANÇA 2: Usar os dados reamostrados

wf.close()

In [ ]:
import IPython.display as ipd
ipd.Audio("output.wav")

## Teste Modelo:

In [ ]:
!pip install gpiozero

In [ ]:
!pip install tflite-runtime

In [ ]:
!pip install librosa

In [ ]:
!ls

In [ ]:
import time
import numpy as np
import pyaudio
import librosa
import tflite_runtime.interpreter as tflite
from collections import deque
from scipy.signal import resample_poly
from gpiozero import LED

CLASS_NAMES = ['_silence_', '_unknown_', 'go', 'no', 'off', 'on', 'stop']

MODEL_PATH = "./model_mfcc_fp16.tflite"

NATIVE_RATE = 48000
TARGET_RATE = 16000
AUDIO_DURATION_S = 1
SAMPLES_PER_AUDIO_TARGET = TARGET_RATE * AUDIO_DURATION_S
SAMPLES_PER_AUDIO_NATIVE = NATIVE_RATE * AUDIO_DURATION_S

CHUNK = 1024
DEVICE_INDEX = 0
CHANNELS = 1
FORMAT = pyaudio.paInt16

N_FFT = 512
FRAME_STEP = 256
N_MELS = 128
N_MFCCS = 40
F_MIN = 20
F_MAX = TARGET_RATE / 2

print("Carregando modelo TFLite...")
interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Modelo carregado com sucesso.")
print("Input shape:", input_details[0]['shape'])

# --- 2. FUNÇÃO DE PRÉ-PROCESSAMENTO (CORRIGIDA) ---
def preprocess_audio(audio_data):
    """
    Recebe 1 segundo de áudio, extrai os MFCCs e formata
    para a entrada do modelo.
    """
    mfccs = librosa.feature.mfcc(
        y=audio_data, sr=TARGET_RATE, n_mfcc=N_MFCCS, n_fft=N_FFT,
        hop_length=FRAME_STEP, win_length=N_FFT, n_mels=N_MELS,
        fmin=F_MIN, fmax=F_MAX, center=False
    )
    
    # --- LINHA ADICIONADA AQUI ---
    # Transpõe a matriz para (tempo, features) -> (61, 40)
    mfccs = mfccs.T
    
    # Adiciona a dimensão do canal (necessário para modelos CNN)
    mfccs = np.expand_dims(mfccs, axis=-1)
    
    # Adiciona a dimensão do batch (o modelo espera um lote de dados)
    mfccs = np.expand_dims(mfccs, axis=0)
    
    return mfccs.astype(np.float32)

# --- 3. LOOP DE INFERÊNCIA EM TEMPO REAL ---
audio = pyaudio.PyAudio()
stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=NATIVE_RATE,
    input=True,
    input_device_index=DEVICE_INDEX,
    frames_per_buffer=CHUNK
)

audio_buffer = deque(maxlen=SAMPLES_PER_AUDIO_NATIVE)

inference_trigger_count = 40 # Dispara a cada ~0.5 segundos (20 * 1024 / 48000)
chunk_counter = 0

print("\n--- Pressione CTRL+C para parar ---")
print("Ouvindo...")

try:
    while True:
        data = stream.read(CHUNK, exception_on_overflow=False)
        chunk_data = np.frombuffer(data, dtype=np.int16)
        audio_buffer.extend(chunk_data)
        chunk_counter += 1

        # Gatilho de inferência: executa quando o contador atinge o limite E o buffer está cheio
        if chunk_counter >= inference_trigger_count and len(audio_buffer) == SAMPLES_PER_AUDIO_NATIVE:
            # Pega o snapshot atual do buffer (1 segundo de áudio a 48kHz)
            waveform_native = np.array(audio_buffer, dtype=np.float32)
            
            # --- REAMOSTRAGEM (RESAMPLING) ---
            waveform_resampled = resample_poly(waveform_native, TARGET_RATE, NATIVE_RATE)
            waveform_resampled = waveform_resampled.astype(np.float32) # Garante float32
            
            # Normaliza o áudio para o intervalo [-1, 1]
            waveform_final = waveform_resampled / 32768.0

            # Realiza o pré-processamento
            input_data = preprocess_audio(waveform_final)

            # Executa a inferência
            interpreter.set_tensor(input_details[0]['index'], input_data)
            interpreter.invoke()
            
            output_data = interpreter.get_tensor(output_details[0]['index'])
            prediction_index = np.argmax(output_data[0])
            predicted_class = CLASS_NAMES[prediction_index]
            confidence = np.max(output_data[0])

            if predicted_class != '_unknown_' and confidence > 0.9:
                print(f"DETECÇÃO: {predicted_class} (Confiança: {confidence:.2f})")
            else:
                pass
            
            # Reseta o contador
            chunk_counter = 0

except KeyboardInterrupt:
    print("\nParando a execução...")

finally:
    stream.stop_stream()
    stream.close()
    audio.terminate()
    print("Recursos liberados.")


## Teste LEDs: